# C01 — Modelos LLM

## Problema escolhido: Análise de Feedbacks de Clientes

Empresas recebem grandes volumes de avaliações em texto livre. Este notebook demonstra como
modelos pré-treinados podem automatizar tarefas de linguagem natural sobre esse corpus:

| Tarefa | Modelo | Arquitetura |
|---|---|---|
| Classificação de sentimento | DistilBERT (HuggingFace) | Encoder-only |
| Geração de resposta automática | DistilGPT-2 (HuggingFace) | Decoder-only |
| Sumarização de múltiplos feedbacks | FLAN-T5-small (HuggingFace) | Encoder-Decoder |
| Reescrita/padronização de texto | Claude Haiku (Anthropic API) | Decoder-only (remoto) |

## 0. Instalação de dependências

In [7]:
%pip install transformers torch anthropic python-dotenv sentencepiece -q


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 1. Corpus de feedbacks de clientes

Conjunto representativo com sentimentos positivos, negativos e neutros.

In [ ]:
from pathlib import Path

DATA_DIR = Path("data")

def cargar_feedbacks(arquivo: str) -> list[str]:
    return [l.strip() for l in (DATA_DIR / arquivo).read_text().splitlines() if l.strip()]

feedbacks_pos  = cargar_feedbacks("feedbacks_positivos.txt")
feedbacks_neg  = cargar_feedbacks("feedbacks_negativos.txt")
feedbacks_neut = cargar_feedbacks("feedbacks_neutros.txt")

FEEDBACKS = feedbacks_pos + feedbacks_neg + feedbacks_neut
print(f"{len(FEEDBACKS)} feedbacks carregados ({len(feedbacks_pos)} pos / "
      f"{len(feedbacks_neg)} neg / {len(feedbacks_neut)} neutros).")

---
## 2. Tarefa 1 — Classificação de sentimento (Encoder-only: DistilBERT)

### Arquitetura: Encoder-only
Modelos como BERT e DistilBERT processam o texto **bidireccionalmente** — cada token
"vê" todos os outros ao mesmo tempo. Isso os torna ideais para tarefas de **compreensão**
(classificação, extração de entidades, QA extrativo), mas **não conseguem gerar texto novo**.

- Representação: token `[CLS]` captura o significado global da sequência.
- DistilBERT é uma versão destilada do BERT: 40% menor, 60% mais rápido, 97% da performance.

In [9]:
from transformers import pipeline

# Pipeline de sentimento — usa distilbert-base-uncased-finetuned-sst-2-english por padrão
classificador = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

resultados = classificador(FEEDBACKS)

print("Classificação de Sentimento\n" + "=" * 50)
for feedback, resultado in zip(FEEDBACKS, resultados):
    label = resultado["label"]
    score = resultado["score"]
    emoji = "✓" if label == "POSITIVE" else "✗"
    print(f"{emoji} [{label:8s} {score:.2f}] {feedback[:60]}...")

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Classificação de Sentimento
✓ [POSITIVE 1.00] The product arrived on time and works perfectly. Very satisf...
✗ [NEGATIVE 1.00] Terrible experience. The item broke after two days and custo...
✗ [NEGATIVE 0.99] Average product. Does what it promises but nothing extraordi...
✓ [POSITIVE 1.00] Excellent quality! Exceeded my expectations. Will definitely...
✗ [NEGATIVE 1.00] The packaging was damaged and one piece was missing from the...
✓ [POSITIVE 0.98] Fast shipping, good price. The product is decent but instruc...


### 2.1 Inspecionando a tokenização (encoder-only)

O tokenizador converte texto em IDs numéricos. Tokens especiais `[CLS]` e `[SEP]` são
inseridos automaticamente pelo modelo BERT.

In [10]:
from transformers import AutoTokenizer

tok_bert = AutoTokenizer.from_pretrained("distilbert-base-uncased-finetuned-sst-2-english")

exemplo = FEEDBACKS[0]
tokens = tok_bert.tokenize(exemplo)
ids = tok_bert.encode(exemplo)

print(f"Texto original : {exemplo}")
print(f"Tokens ({len(tokens)}) : {tokens}")
print(f"IDs (com especiais): {ids}")
print(f"Vocabulário total: {tok_bert.vocab_size:,} tokens")

Texto original : The product arrived on time and works perfectly. Very satisfied with the purchase!
Tokens (15) : ['the', 'product', 'arrived', 'on', 'time', 'and', 'works', 'perfectly', '.', 'very', 'satisfied', 'with', 'the', 'purchase', '!']
IDs (com especiais): [101, 1996, 4031, 3369, 2006, 2051, 1998, 2573, 6669, 1012, 2200, 8510, 2007, 1996, 5309, 999, 102]
Vocabulário total: 30,522 tokens


---
## 3. Tarefa 2 — Geração de resposta automática (Decoder-only: DistilGPT-2)

### Arquitetura: Decoder-only
Modelos como GPT-2, Llama e Claude usam **atenção causal** — cada token só "vê" os
tokens anteriores. São autoregressivos: geram um token de cada vez, condicionados no
que veio antes. Ideais para **geração de texto**, mas menos eficientes para compreensão
pura que os encoders.

### Parâmetros de geração (decisões importantes)
| Parâmetro | Efeito |
|---|---|
| `max_new_tokens` | Limita o tamanho da saída |
| `temperature` | > 1 = mais criativo; < 1 = mais determinístico |
| `top_p` | Nucleus sampling — filtra tokens improváveis |
| `do_sample` | False = greedy decoding (sempre o token mais provável) |

In [11]:
gerador = pipeline(
    "text-generation",
    model="distilgpt2",
    pad_token_id=50256  # evita warning de padding
)

# Feedback negativo — gerar rascunho de resposta do suporte
feedback_negativo = FEEDBACKS[1]
prompt = f"Customer complaint: {feedback_negativo}\nSupport response:"

resposta = gerador(
    prompt,
    max_new_tokens=60,
    temperature=0.7,
    top_p=0.9,
    do_sample=True,
    num_return_sequences=1
)

texto_gerado = resposta[0]["generated_text"]
apenas_resposta = texto_gerado[len(prompt):].strip()

print(f"Feedback   : {feedback_negativo}")
print(f"Resposta   : {apenas_resposta}")

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'pad_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Passing `generation_config` together with generation-related arguments=({'num_return_sequences', 'top_p', 'max_new_tokens', 'temperature', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=60) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed 

Feedback   : Terrible experience. The item broke after two days and customer support never responded.
Resposta   : Thank you, Terrible customer service. The item broke after two days and customer support never responded.
Please send me a message if you have any questions.
Thank you, Terrible customer service. The item broke after two days and customer support never responded.
Thank you, Terrible customer


### 3.1 Efeito do parâmetro `temperature`

In [12]:
prompt_curto = "The product broke after two days. Support response:"

for temp in [0.3, 0.7, 1.2]:
    saida = gerador(
        prompt_curto,
        max_new_tokens=30,
        temperature=temp,
        do_sample=True
    )
    texto = saida[0]["generated_text"][len(prompt_curto):].strip()
    print(f"temperature={temp}: {texto}")

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=30) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=30) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


temperature=0.3: 


[transformers] Both `max_new_tokens` (=30) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


temperature=0.7: 
temperature=1.2: We are continuing to investigate the impact of this breach of trust on your website. The latest information will include contact information. We are in compliance - we


---
## 4. Tarefa 3 — Sumarização (Encoder-Decoder: FLAN-T5-small)

### Arquitetura: Encoder-Decoder (Seq2Seq)
Modelos como T5 e BART combinam um **encoder bidirecional** (para compreender a entrada)
com um **decoder autoregressivo** (para gerar a saída). Adequados para tarefas de
**transformação de texto**: sumarização, tradução, QA generativo.

FLAN-T5 é fine-tuned com instruções em linguagem natural — basta prefixar a tarefa.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

model_id = "google/flan-t5-small"
tok_t5   = AutoTokenizer.from_pretrained(model_id)
model_t5 = AutoModelForSeq2SeqLM.from_pretrained(model_id)

def flan_generate(prompt: str, max_new_tokens: int = 80) -> str:
    inputs = tok_t5(prompt, return_tensors="pt", truncation=True, max_length=512)
    with torch.no_grad():
        out = model_t5.generate(**inputs, max_new_tokens=max_new_tokens)
    return tok_t5.decode(out[0], skip_special_tokens=True)

todos_feedbacks = " | ".join(FEEDBACKS)
instrucao = f"Summarize the main customer issues and praises from these reviews: {todos_feedbacks}"
print("Resumo dos feedbacks:")
print(flan_generate(instrucao))

### 4.1 Question Answering sobre os feedbacks

In [ ]:
perguntas = [
    "What are the main complaints from customers?",
    "What do customers praise most?",
    "How many customers mentioned shipping?",
]

for pergunta in perguntas:
    instrucao_qa = f"Based on these reviews: {todos_feedbacks}\n\nAnswer: {pergunta}"
    resposta = flan_generate(instrucao_qa, max_new_tokens=50)
    print(f"Q: {pergunta}")
    print(f"A: {resposta}\n")

---
## 5. Tarefa 4 — Reescrita e padronização (API remota: Claude Haiku)

Para tarefas que exigem **alta qualidade de linguagem**, instrução complexa ou contexto
longo, a API remota oferece capacidade muito superior aos modelos leves locais.

**Justificativa do provedor:** Claude Haiku é um modelo decoder-only de grande escala com
excelente razão custo/qualidade para tarefas de produção. Não requer GPU local.

In [ ]:
import os
import anthropic
from dotenv import load_dotenv

load_dotenv()
client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

SYSTEM_REESCRITA = """
Você é um especialista em CX (Customer Experience).
Reescreva feedbacks de clientes no formato padronizado:
- Sentimento: [POSITIVO | NEGATIVO | NEUTRO]
- Problema/Elogio principal: (1 frase)
- Urgência: [ALTA | MÉDIA | BAIXA]
- Ação recomendada: (1 frase)
"""

for i, feedback in enumerate(FEEDBACKS[:3], 1):  # primeiros 3 para economizar tokens
    resp = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=200,
        system=SYSTEM_REESCRITA,
        messages=[{"role": "user", "content": feedback}]
    )
    print(f"--- Feedback {i} ---")
    print(f"Original  : {feedback}")
    print(f"Padronizado:\n{resp.content[0].text}")
    print()

---
## 6. Comparação de tokenizadores

Cada modelo usa sua própria tokenização. As diferenças afetam o número de tokens
consumidos e o custo (em modelos via API).

In [ ]:
from transformers import AutoTokenizer

modelos_tok = {
    "DistilBERT (encoder)": "distilbert-base-uncased-finetuned-sst-2-english",
    "DistilGPT-2 (decoder)": "distilgpt2",
    "FLAN-T5-small (enc-dec)": "google/flan-t5-small",
}

texto = FEEDBACKS[0]
print(f"Texto: {texto}\n")
print(f"{'Modelo':<30} {'Vocab':>8} {'Tokens':>8} {'Tokens especiais'}")
print("-" * 70)

for nome, model_id in modelos_tok.items():
    tok = AutoTokenizer.from_pretrained(model_id)
    ids = tok.encode(texto)
    especiais = [tok.convert_ids_to_tokens(i) for i in ids if i in tok.all_special_ids]
    print(f"{nome:<30} {tok.vocab_size:>8,} {len(ids):>8} {especiais}")

---
## 7. Discussão: escolhas de modelo, limitações e adequação

### 7.1 Encoder-only vs Decoder-only vs Encoder-Decoder

In [ ]:
import pandas as pd

discussao = pd.DataFrame([
    {
        "Arquitetura": "Encoder-only (DistilBERT)",
        "Atenção": "Bidirecional",
        "Ideal para": "Classificação, NER, QA extrativo",
        "Não adequado para": "Geração de texto livre",
        "Limitação observada": "Não gera respostas, só classifica",
    },
    {
        "Arquitetura": "Decoder-only (DistilGPT-2)",
        "Atenção": "Causal (unidirecional)",
        "Ideal para": "Geração de texto, completação",
        "Não adequado para": "Compreensão profunda, classificação",
        "Limitação observada": "Respostas incoerentes sem fine-tuning para suporte",
    },
    {
        "Arquitetura": "Enc-Dec (FLAN-T5-small)",
        "Atenção": "Bidirecional + Causal",
        "Ideal para": "Sumarização, tradução, QA generativo",
        "Não adequado para": "Geração longa e criativa",
        "Limitação observada": "Modelo small gera respostas curtas/superficiais",
    },
    {
        "Arquitetura": "Decoder-only (Claude Haiku — API)",
        "Atenção": "Causal (grande escala)",
        "Ideal para": "Instrução complexa, reescrita, padronização",
        "Não adequado para": "Uso offline ou alto volume sem custo",
        "Limitação observada": "Custo por token, dependência de internet",
    },
])

pd.set_option("display.max_colwidth", 50)
discussao

### 7.2 Conclusão

Para o problema de **análise de feedbacks de clientes**, a abordagem híbrida é a mais eficiente:

1. **DistilBERT** — triagem rápida e barata de sentimento em alto volume (local, sem custo por chamada).
2. **FLAN-T5** — sumarização de lotes de feedbacks para relatórios periódicos (local).
3. **Claude via API** — padronização e resposta reutilizável para casos de alta urgência (melhor qualidade, custo controlado).
4. **DistilGPT-2** — inadequado para produção neste caso de uso sem fine-tuning específico de suporte ao cliente.

A escolha do modelo deve equilibrar **qualidade, latência, custo e disponibilidade offline**.

---
## 8. Uso de tokens (API Anthropic)

In [ ]:
# Demonstração de contagem de tokens antes de enviar (evita custos inesperados)
conteudo = [{"role": "user", "content": FEEDBACKS[0]}]

contagem = client.messages.count_tokens(
    model="claude-haiku-4-5-20251001",
    system=SYSTEM_REESCRITA,
    messages=conteudo
)

print(f"Tokens de entrada estimados: {contagem.input_tokens}")
print("(Útil para estimar custo antes de executar chamadas em lote)")

---
## 9. Análisis de la transcripción en español

Aplicamos los mismos modelos sobre la transcripción de clase `data/raw/*_espanhol.txt`.
La transcripción está en formato WebVTT — primero la limpiamos y luego tokenizamos,
generamos respuestas y guardamos un resumen en disco.

In [ ]:
# Celda A — tokenización del transcript con IDs y cuenta de tokens
import re
from pathlib import Path
from transformers import AutoTokenizer

transcript_path = Path("data/raw/Sistemas_Cognitivos_com_LLMs_aula-17-06-2026_20-05_espanhol.txt")
raw_text = transcript_path.read_text(encoding="utf-8")

lines = [
    l.strip() for l in raw_text.splitlines()
    if l.strip()
    and not l.startswith("WEBVTT")
    and not re.match(r'^\d+$', l.strip())
    and not re.match(r'^\d{2}:\d{2}', l.strip())
]
clean_transcript = " ".join(lines)
fragmento = clean_transcript[:1500]

tok_gpt2 = AutoTokenizer.from_pretrained("distilgpt2")
ids = tok_gpt2.encode(fragmento)

print(f"Transcripción limpia (primeros 300 chars):\n{fragmento[:300]}")
print(f"\nIDs (primeros 20): {ids[:20]}")
print(f"Total tokens en fragmento: {len(ids)}")

In [ ]:
# Celda B — 3 respuestas coherentes con contenido de la transcripción
prompt_transcript = f"Class summary: {fragmento[:400]}\nKey insight:"

respuestas = gerador(
    prompt_transcript,
    max_new_tokens=50,
    temperature=0.7,
    do_sample=True,
    num_return_sequences=3,
    pad_token_id=50256,
)

print("3 respuestas generadas a partir del transcript:\n")
for i, r in enumerate(respuestas, 1):
    texto = r["generated_text"][len(prompt_transcript):].strip()
    print(f"Respuesta {i}: {texto}\n")

In [ ]:
# Celda C — resumen guardado en data/processed/summary_espanhol.md
resumen_prompt = (
    f"Summarize this lecture transcript in Spanish in 3-5 sentences: {fragmento[:800]}"
)
resumen_texto = flan_generate(resumen_prompt, max_new_tokens=120)

output_dir = Path("data/processed")
output_dir.mkdir(exist_ok=True)
output_file = output_dir / "summary_espanhol.md"
output_file.write_text(f"# Resumen de la clase\n\n{resumen_texto}\n", encoding="utf-8")

print(f"Resumen guardado en: {output_file}")
print(f"\nContenido:\n{resumen_texto}")